# 05. Field-visit plan for January

Plans where to look in Paoay so the January visit can check the improved map and supply the first local labels. Stratified random points by map class (carrot, cabbage, potato, Unknown) inside land-activity cropland, at least 50 m apart, with as many reserves as primary points; plus a few points inside every KMZ reference area (not used for accuracy). After the visit, the filled field sheet gives stratum-weighted accuracy of the map's crop labels.

**Input:** `current_paoay.json` (notebook 04). **Output:** `current_field_plan.json`, `visit_points.csv`, `visit_points.kml` (Google Earth / phone), `visit_points.geojson`, `field_sheet.csv`. Predictions stay in `visit_points.csv` for the analysis; the KML and field sheet leave them out to avoid observation bias.

## Setup

In [ ]:
from pathlib import Path
import json
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").is_file():
    if root == root.parent:
        raise RuntimeError("Open this notebook inside the project repository")
    root = root.parent
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
if str(root) not in sys.path:
    sys.path.insert(0, str(root))  # model_aza

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from IPython.display import display
from model_aza.improved import field_plan, workflow
pd.set_option("display.width", 200)

## Build the plan

In [ ]:
paoay_handoff = ""  # Optional Paoay receipt; blank uses the current pointer
seed = 1
spacing_m = 50
receipt = field_plan.build_plan(root, paoay_handoff or None, seed, spacing_m)
run = receipt.parent
plan = json.loads((run / "plan.json").read_text())
points = pd.read_csv(run / "visit_points.csv")
print("Plan receipt:", receipt)
display(pd.Series(plan["stratum_pixels"], name="cropland pixels in stratum").to_frame())
display(points.pivot_table(index="map_label", columns="group", values="visit_id", aggfunc="count", fill_value=0))

## Map with visit points

In [ ]:
import rasterio
paoay = Path(json.loads((root / "pipeline_outputs/improved/current_paoay.json").read_text())["path"]).parent
with rasterio.open(paoay / "crop_map.tif") as raster:
    codes = raster.read(1)
colors = ListedColormap(["#ffffff", "#ff8c00", "#9acd60", "#7a5444", "#d9d9d9", "#f2f2f2"])
figure, axis = plt.subplots(figsize=(10, 9))
axis.imshow(codes, cmap=colors, vmin=0, vmax=5, interpolation="nearest")
style = {"stratified": ("o", "#d62728", "Stratified (primary)"), "reference_area": ("s", "#1f77b4", "Reference area"),
         "reserve": ("x", "#555555", "Reserve")}
for group, (marker, color, name) in style.items():
    chosen = points[points.group == group]
    axis.scatter(chosen.col, chosen.row, marker=marker, s=28, c=color, label=f"{name} ({len(chosen)})")
for _, p in points[points.group != "reserve"].iterrows():
    axis.annotate(p.visit_id, (p.col, p.row), fontsize=5, xytext=(2, 2), textcoords="offset points")
handles, labels = axis.get_legend_handles_labels()
handles += [plt.Rectangle((0, 0), 1, 1, color=colors(i)) for i in (1, 2, 3, 4)]
labels += ["Map: carrot", "Map: cabbage", "Map: potato", "Map: Unknown"]
axis.legend(handles, labels, loc="lower left", fontsize=8, frameon=True)
axis.set_title("Paoay field-visit plan (improved map; exploratory)")
axis.axis("off")
figure.tight_layout()
figure.savefig(run / "visit_plan.png", dpi=150)
display(figure)
plt.close(figure)

## Visit list

In [ ]:
display(points[["visit_id", "group", "map_label", "top_probability", "reference_area", "latitude", "longitude"]].round(3))

## After the visit: accuracy from the field sheet

Take `visit_points.kml` and `field_sheet.csv` to the field; neither shows the map's prediction, so record what is there first. The sheet has the same columns as Paoay notebook 05's `observations.csv`, plus visit ID and group. When a primary point is inaccessible, use the next reserve of the same stratum and change its `group` to `stratified`.

In [ ]:
import csv
with (run / "field_sheet.csv").open() as stream:
    sheet = list(csv.DictReader(stream))
filled = [r for r in sheet if (r.get("surface_state") or "").strip()]
if filled:
    display(field_plan.estimate_accuracy(sheet, plan["stratum_pixels"], points.to_dict("records")))
else:
    print(f"No observations yet: {len(sheet)} blank rows in {run / 'field_sheet.csv'}")